![SGSSS Logo](https://github.com/SGSSSonline/text-analysis/blob/main/img/SGSSS_Stacked.png?raw=true)

# Practical 1: Preprocessing Text

Computational text analysis allows social scientists to study language at scale — from parliamentary debates and policy documents to social media posts and interview transcripts. But before any analysis can take place, raw text must be transformed into a structured, numerical format that a computer can work with.

This practical introduces the foundational preprocessing steps that underpin nearly all text analysis workflows. We will work through each step using a sample of UK parliamentary speeches (Hansard debates), transforming messy, unstructured text into a clean document-feature matrix ready for analysis.

## Aims

This practical has two aims:

1. **Demonstrate how to use R to preprocess text data** for social science research.
2. **Cultivate computational thinking skills** — breaking down a text preprocessing problem into a series of logical, repeatable steps.

### Lesson Details

| | |
| --- | --- |
| **Level** | Introductory |
| **Time** | ~30 minutes |
| **Pre-requisites** | None |
| **Learning outcomes** | Understand the key steps involved in preprocessing text data |
| | Be able to tokenise text using R |
| | Be able to apply lowercasing, punctuation removal, and stopword removal |
| | Understand stemming and how it reduces words to a common root |
| | Be able to construct a document-feature matrix |
| | Understand TF-IDF weighting and its purpose |

## Guide to Using This Resource

This is a **Jupyter Notebook** — an interactive document that combines text, code, and output in a single environment. If you are viewing this in **Google Colab**, you are running the notebook in the cloud, which means you do not need to install anything on your own machine.

**Note: This notebook uses R.** In Google Colab, you need to change the runtime: go to **Runtime > Change runtime type > select R**.

A notebook is made up of **cells**. There are two main types:

- **Markdown cells** contain formatted text (like this one). They provide explanations, instructions, and context.
- **Code cells** contain R code that you can execute. Code cells are displayed with a grey background and have a play button on the left.

To **run a cell**, click on it and press `Shift+Enter` (or click the play button). The output will appear directly below the cell. You should run the code cells **in order**, from top to bottom, as later cells often depend on variables or packages loaded in earlier cells.

If you are using **Google Colab**, make sure to save a copy of this notebook to your own Google Drive first: go to **File > Save a copy in Drive**.

If you are new to Jupyter Notebooks and would like a more detailed introduction, see the excellent materials by Dani Arribas-Bel: [https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb](https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb)

### Interactive Test

Let's make sure everything is working. Run the cell below and enter your name when prompted.

In [ ]:
cat("Hello! What is your name?\n")
name <- readline()
cat(paste0("Welcome to the course, ", name, "!\n"))

## Setup

Before we begin, we need to install and load the R packages we will use throughout this practical. These include:

- **quanteda** — a comprehensive package for managing and analysing text data. It provides functions for tokenisation, stopword removal, stemming, and constructing document-feature matrices.
- **quanteda.textstats** — for computing text statistics such as term frequencies.
- **quanteda.textplots** — for visualising text data.

In [ ]:
# Install packages (if needed on Colab)
install.packages(c("quanteda", "quanteda.textstats", "quanteda.textplots"), quiet = TRUE)

library(quanteda)
library(quanteda.textstats)
library(quanteda.textplots)

cat("Successfully loaded packages\n")

## Loading the Corpus

We will work with a small sample of UK parliamentary speeches drawn from the **Hansard debates** — the official record of proceedings in the UK Parliament. Parliamentary speech data is widely used in social science research for studying political communication, party positioning, policy framing, and more.

The dataset below contains 20 speech excerpts with metadata including the speaker's name, party affiliation, and the date of the speech. These excerpts are illustrative and cover a range of policy topics debated in the House of Commons.

In [ ]:
# Create a sample dataset of parliamentary speech excerpts
df <- data.frame(
  date = c(
    "2024-01-15", "2024-01-15", "2024-01-22", "2024-01-22", "2024-02-05",
    "2024-02-05", "2024-02-12", "2024-02-12", "2024-02-19", "2024-02-19",
    "2024-03-04", "2024-03-04", "2024-03-11", "2024-03-11", "2024-03-18",
    "2024-03-18", "2024-03-25", "2024-03-25", "2024-04-01", "2024-04-01"
  ),
  speaker = c(
    "Margaret Thornton", "James Whitfield", "Sarah Okonkwo", "David Hargreaves",
    "Emily Chen", "Robert MacLeod", "Fatima Hussain", "Thomas Brennan",
    "Catherine Lloyd", "Andrew Patel", "Helen Murray", "Kwame Asante",
    "Patricia Gallagher", "Ravi Sharma", "Fiona Blackwood", "Marcus Johnson",
    "Alison Crawford", "Yusuf Ali", "Diana Novak", "Christopher Reeves"
  ),
  party = c(
    "Labour", "Conservative", "Labour", "Conservative", "Labour",
    "SNP", "Labour", "Conservative", "Liberal Democrat", "Labour",
    "SNP", "Labour", "Conservative", "Labour", "SNP",
    "Conservative", "Labour", "Labour", "Liberal Democrat", "Conservative"
  ),
  speech_text = c(
    "The National Health Service remains the cornerstone of our social contract with the British people. We must invest in frontline services and address the chronic staffing shortages that are putting patients at risk. The waiting lists have grown to record levels and this government must take urgent action to reduce them.",
    "Economic growth is the foundation upon which all public services depend. By cutting taxes and reducing unnecessary regulation, we can unleash the potential of British businesses. The private sector, not the state, is the engine of prosperity and job creation in this country.",
    "The climate crisis demands immediate and ambitious action from this House. We cannot afford to delay investment in renewable energy infrastructure. Our children and grandchildren will judge us by the decisions we make today on carbon emissions and environmental protection.",
    "Immigration policy must be fair but firm. We need a points-based system that attracts the skilled workers our economy needs while maintaining control of our borders. The British people voted for sovereignty and we must deliver on that promise.",
    "Education is the great equaliser in our society. Every child, regardless of their background or postcode, deserves access to excellent teaching and modern facilities. We must close the attainment gap between the most and least disadvantaged pupils in our schools.",
    "Scotland's interests are consistently overlooked by this Westminster government. The devolution settlement is being undermined at every turn. The people of Scotland deserve the right to determine their own future and to have their voice heard on matters that affect their daily lives.",
    "The housing crisis is devastating communities across the country. Young people cannot afford to buy their first home and rents are consuming an ever larger share of household incomes. We need a massive programme of social housing construction to address this emergency.",
    "Defence spending must remain a top priority for this government. The threats we face from hostile state actors are growing more complex and more dangerous. We must ensure our armed forces have the equipment and resources they need to keep this nation safe.",
    "Mental health services are chronically underfunded and overstretched. Too many people are waiting months or even years for treatment they desperately need. Parity of esteem between physical and mental health must become a reality, not just a slogan.",
    "The cost of living crisis is hitting working families hardest. Energy bills have skyrocketed and food prices continue to rise at alarming rates. The government must do more to support those who are struggling to heat their homes and feed their children.",
    "The fishing communities of Scotland have been betrayed by broken promises on Brexit. Access to our waters and fair quota allocations are essential for the survival of these coastal communities. We will continue to fight for the rights of Scottish fishermen in this House.",
    "Public transport infrastructure is failing passengers across the north of England. Years of underinvestment have left communities isolated and reliant on expensive and unreliable services. We need a transport revolution that connects people to jobs, education, and opportunity.",
    "Law and order must be restored to our streets. Violent crime and antisocial behaviour are blighting communities and the police need the resources and powers to tackle these problems effectively. We will always stand on the side of victims and law-abiding citizens.",
    "The arts and creative industries contribute billions to our economy and enrich the cultural life of this nation. Cuts to arts funding have devastated regional theatres, museums, and music venues. Investment in culture is not a luxury; it is an investment in our communities and our identity.",
    "Rural communities in Scotland face unique challenges that this government fails to understand. Broadband connectivity, access to healthcare, and affordable transport are not luxuries but necessities. The centralisation of services in urban areas is leaving rural Scotland behind.",
    "Free trade agreements are essential for our post-Brexit economic strategy. British goods and services must have access to global markets on the most favourable terms possible. We are building new trading relationships that will benefit every region of the United Kingdom.",
    "Workers' rights must be strengthened, not eroded, in the modern economy. The rise of zero-hours contracts and the gig economy has created a new class of precarious employment. Every worker deserves fair pay, decent conditions, and the security of knowing their rights are protected by law.",
    "Child poverty is a stain on our national conscience. Over four million children in this country are growing up in poverty, and the numbers are rising. We must reform the welfare system to ensure that no child goes hungry and every family has a decent standard of living.",
    "Local government is the backbone of democracy in this country. Councils are being asked to do more with less, and essential services are being cut to the bone. Proper funding for local authorities is not just desirable; it is essential for the health of our democratic institutions.",
    "Science and innovation are the keys to our future prosperity. The United Kingdom has world-leading universities and research institutions that must be supported and celebrated. Government investment in research and development is an investment in the jobs and industries of tomorrow."
  ),
  stringsAsFactors = FALSE
)

cat(paste0("Dataset shape: ", nrow(df), " speeches, ", ncol(df), " columns\n"))
head(df)

Let's take a closer look at one of the speeches to understand what we are working with.

In [ ]:
# Select a single speech to use as a working example
sample_text <- df$speech_text[1]
cat(paste0("Speaker: ", df$speaker[1], " (", df$party[1], ")\n"))
cat(paste0("Date: ", df$date[1], "\n"))
cat(paste0("\nSpeech text:\n", sample_text, "\n"))

## Creating a quanteda Corpus

Before we can tokenise or preprocess the text, we need to create a **corpus** object. In the `quanteda` package, a corpus is a structured collection of texts along with their associated metadata (called "docvars" or document variables).

We create the corpus from the `speech_text` column of our data frame and attach the metadata columns (speaker, party, date) as document variables. This allows us to filter, group, and analyse the texts by their metadata later on.

In [ ]:
# Create a quanteda corpus from the speech_text column
corp <- corpus(df, text_field = "speech_text")

# Summarise the corpus
cat(paste0("Number of documents: ", ndoc(corp), "\n"))
summary(corp)

The corpus summary shows each document along with its metadata. The `Types` column shows the number of unique word forms, `Tokens` shows the total number of words, and `Sentences` shows the number of sentences in each document. These counts are based on the raw text before any preprocessing.

## Tokenisation

The first step in preprocessing text is **tokenisation** — splitting text into individual units called **tokens**, which are usually words. This is the foundation of all subsequent text analysis: once we have a list of individual tokens, we can count them, filter them, and transform them.

In `quanteda`, we use the `tokens()` function. By default, it splits text on whitespace and punctuation boundaries, producing a list of word tokens for each document.

In [ ]:
# Tokenise the sample speech
sample_tokens <- tokens(sample_text)

# Display the tokens
cat("Tokens:\n")
print(sample_tokens)
cat(paste0("\nNumber of tokens: ", ntoken(sample_tokens), "\n"))
cat(paste0("Number of unique tokens: ", ntype(sample_tokens), "\n"))

Notice that the tokeniser has split the text into individual words **and** punctuation marks. Punctuation like full stops and commas are treated as separate tokens. This is expected — we will deal with punctuation in the next step.

Also notice the difference between the total number of tokens and the number of unique tokens. This tells us that some words appear more than once in the speech.

## Lowercasing and Punctuation Removal

The next step is to **reduce complexity** in the text. Two common transformations are:

1. **Lowercasing**: Converting all tokens to lowercase so that "The" and "the" are treated as the same word. Unless capitalisation is of analytical interest, this is standard practice.
2. **Punctuation removal**: Removing punctuation marks (commas, full stops, etc.) that are not substantively informative for most text analysis tasks.

In `quanteda`, we can combine these steps: `tokens()` has a `remove_punct` argument, and `tokens_tolower()` converts all tokens to lowercase.

In [ ]:
# Remove punctuation and convert to lowercase
sample_tokens_clean <- tokens(sample_text, remove_punct = TRUE, remove_numbers = TRUE)
sample_tokens_clean <- tokens_tolower(sample_tokens_clean)

cat("Before cleaning:\n")
cat(paste0("  Tokens: ", ntoken(sample_tokens), "\n"))
cat(paste0("  Unique: ", ntype(sample_tokens), "\n"))

cat("\nAfter lowercasing and punctuation removal:\n")
cat(paste0("  Tokens: ", ntoken(sample_tokens_clean), "\n"))
cat(paste0("  Unique: ", ntype(sample_tokens_clean), "\n"))

print(sample_tokens_clean)

We can see that lowercasing and punctuation removal have reduced both the total number of tokens and the number of unique tokens. This is good — we have removed noise without losing substantive content.

## Stopword Removal

**Stopwords** are common words that appear frequently in any text but carry little substantive meaning — words like "the", "is", "and", "of", "to". These words are essential for grammar but are rarely informative for content analysis.

`quanteda` provides a built-in list of English stopwords via the `stopwords("en")` function. Removing them helps us focus on the words that actually distinguish one document from another.

In [ ]:
# View some of the stopwords
en_stopwords <- stopwords("en")
cat(paste0("Number of stopwords in quanteda list: ", length(en_stopwords), "\n"))
cat(paste0("Examples: ", paste(head(sort(en_stopwords), 20), collapse = ", "), "\n"))

In [ ]:
# Remove stopwords from our cleaned tokens
sample_tokens_no_stop <- tokens_remove(sample_tokens_clean, pattern = stopwords("en"))

cat("Before stopword removal:\n")
cat(paste0("  Tokens: ", ntoken(sample_tokens_clean), "\n"))

cat("\nAfter stopword removal:\n")
cat(paste0("  Tokens: ", ntoken(sample_tokens_no_stop), "\n"))

cat(paste0("\nTokens removed: ", ntoken(sample_tokens_clean) - ntoken(sample_tokens_no_stop), "\n"))

print(sample_tokens_no_stop)

Stopword removal typically eliminates a large proportion of tokens. The remaining words are more substantively meaningful — they tell us what the speech is actually *about*.

## Stemming

The final preprocessing step is to create **equivalence classes** — grouping together different forms of the same word so they are counted as one. **Stemming** chops off word endings using simple rules to reduce words to a common root form (called a "stem"). For example, "running", "runs", and "ran" might all be reduced to "run". The result is not always a real word (e.g., "services" becomes "servic").

In `quanteda`, we use `tokens_wordstem()` to apply the Snowball stemmer to our tokens.

**Note**: The Python version of this practical also covers **lemmatisation**, which reduces words to their dictionary form (e.g., "better" maps to "good"). Lemmatisation is not built into `quanteda`. If you need lemmatisation in R, you can use the `textstem` package, but for this practical we will focus on stemming, which is the most common approach in the `quanteda` ecosystem.

In [ ]:
# Apply stemming
sample_tokens_stemmed <- tokens_wordstem(sample_tokens_no_stop)

cat("Before stemming:\n")
cat(paste0("  Tokens: ", ntoken(sample_tokens_no_stop), "\n"))
cat(paste0("  Unique: ", ntype(sample_tokens_no_stop), "\n"))

cat("\nAfter stemming:\n")
cat(paste0("  Tokens: ", ntoken(sample_tokens_stemmed), "\n"))
cat(paste0("  Unique: ", ntype(sample_tokens_stemmed), "\n"))

print(sample_tokens_stemmed)

Let's create a side-by-side comparison to see which words changed after stemming.

In [ ]:
# Compare original and stemmed tokens side by side
original <- unlist(sample_tokens_no_stop)
stemmed <- unlist(sample_tokens_stemmed)

comparison <- data.frame(
  Original = original,
  Stemmed = stemmed,
  stringsAsFactors = FALSE
)

# Show only rows where stemming changed the word
changed <- comparison[comparison$Original != comparison$Stemmed, ]
cat("Words that changed after stemming:\n")
print(changed, row.names = FALSE)

Stemming is more aggressive than lemmatisation, sometimes producing stems that are not real words (e.g., "servic" for "services", "staffing" becomes "staff"). However, it effectively groups related word forms together, which reduces the dimensionality of our data. For the rest of this practical, we will proceed **without** stemming applied to the full corpus, so that results remain easy to interpret. You can always add stemming as an additional step if desired.

## Document-Feature Matrix (DFM)

Now that we know how to preprocess a single document, we can apply these steps to our entire corpus and represent it in a structured numerical format: the **Document-Feature Matrix** (DFM).

A DFM is a table where:
- Each **row** represents a document (in our case, a parliamentary speech).
- Each **column** represents a unique feature (word) in the corpus.
- Each **cell** contains the count of how many times that feature appears in that document.

This is equivalent to what is called a **Document-Term Matrix** (DTM) in other frameworks. It is the standard input format for most text analysis methods. If you are familiar with quantitative data, think of it as a dataset where the variables are words and the observations are documents.

In `quanteda`, we build the DFM in a pipeline: first create the corpus, then tokenise, then apply preprocessing, and finally convert to a DFM using `dfm()`.

In [ ]:
# Build the DFM from the full corpus in a single pipeline
my_dfm <- corp |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower() |>
  tokens_remove(stopwords("en")) |>
  dfm()

cat(paste0("DFM dimensions: ", nrow(my_dfm), " documents x ", ncol(my_dfm), " features\n"))
my_dfm

The DFM is typically **sparse** — most cells contain zeros because any given word only appears in a small number of documents. This is a natural property of language: authors choose from a large vocabulary, so most words are absent from any single document.

Let's see what the most frequent features are across the whole corpus using `topfeatures()`.

In [ ]:
# Find the most common features across all documents
cat("Top 15 most frequent features in the corpus:\n")
topfeatures(my_dfm, n = 15)

## TF-IDF Weighting

Raw word counts are a useful starting point, but they have a limitation: words that are frequent in one document but also frequent across *all* documents are not very informative. For example, if every parliamentary speech mentions "government", then "government" does not help us distinguish one speech from another.

**TF-IDF** (Term Frequency-Inverse Document Frequency) addresses this by weighting words based on two factors:

- **TF (Term Frequency)**: How often a word appears in a specific document. Higher frequency = higher TF.
- **IDF (Inverse Document Frequency)**: How rare a word is across the entire corpus. Words that appear in fewer documents get a higher IDF score.

The TF-IDF score is the product of these two values: **TF-IDF = TF x IDF**. Words that are frequent in one document but rare across the corpus receive the highest scores — these are the words that best characterise what makes a document distinctive.

In `quanteda`, we apply TF-IDF weighting to a DFM using `dfm_tfidf()`.

In [ ]:
# Apply TF-IDF weighting
my_tfidf <- dfm_tfidf(my_dfm)

cat(paste0("TF-IDF matrix dimensions: ", nrow(my_tfidf), " documents x ", ncol(my_tfidf), " features\n"))
my_tfidf

Now let's look at the top features by TF-IDF score for each of the first few speeches. These are the features that are most distinctive to each speech.

In [ ]:
# Display the top 5 features by TF-IDF score for each of the first 5 speeches
for (i in 1:5) {
  speaker <- df$speaker[i]
  party <- df$party[i]
  scores <- as.numeric(my_tfidf[i, ])
  names(scores) <- featnames(my_tfidf)
  top_terms <- sort(scores, decreasing = TRUE)[1:5]
  cat(paste0(speaker, " (", party, "):\n"))
  for (j in seq_along(top_terms)) {
    cat(sprintf("  %-20s %.4f\n", names(top_terms)[j], top_terms[j]))
  }
  cat("\n")
}

The TF-IDF scores highlight the features that are most *distinctive* to each speech. Compare these with the raw frequency counts from the DFM — TF-IDF downweights common words (like "must" or "government") and upweights features that are specific to individual speeches (like "health", "tax", or "climate").

## Exercise

Now it's your turn. Your task is to:

1. **Filter the speeches** to select only those by a specific party (e.g., `"SNP"` or `"Conservative"`).
2. **Create a DFM** for the filtered subset using the same preprocessing steps (tokenise, lowercase, remove punctuation, remove stopwords).
3. **Apply TF-IDF weighting** to the filtered DFM.
4. **Identify the top 10 features** by average TF-IDF score across the selected speeches.
5. **Reflect**: What do these features tell you about the content and priorities of that party's speeches?

Use the skeleton code below to guide you. Replace the `# YOUR CODE HERE` comments with your own code.

**Note**: You can also try using your own text data if you prefer — just create a data frame with a text column and apply the same preprocessing steps.

In [ ]:
# Step 1: Filter speeches by party
party_name <- "SNP"  # Change this to another party if you like
# YOUR CODE HERE: create a filtered data frame called 'party_df'

In [ ]:
# Step 2: Create a corpus and DFM for the filtered speeches
# YOUR CODE HERE: create a corpus, tokenise, clean, and build a DFM

In [ ]:
# Step 3: Apply TF-IDF weighting
# YOUR CODE HERE: use dfm_tfidf() on the filtered DFM

In [ ]:
# Step 4: Identify the top 10 features by average TF-IDF score
# YOUR CODE HERE: calculate the mean TF-IDF score for each feature and sort

**Step 5 — Reflection**: What do the top features tell you about the content and priorities of the party's speeches? Write your answer here.

*YOUR ANSWER HERE*

## Appendix: Exercise Solution

In [ ]:
# --- Exercise Solution ---

# Step 1: Filter speeches by party
party_name <- "SNP"
party_df <- df[df$party == party_name, ]
cat(paste0("Number of ", party_name, " speeches: ", nrow(party_df), "\n"))

In [ ]:
# Step 2: Create a corpus and DFM for the filtered speeches
party_corp <- corpus(party_df, text_field = "speech_text")

party_dfm <- party_corp |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower() |>
  tokens_remove(stopwords("en")) |>
  dfm()

cat(paste0("DFM dimensions: ", nrow(party_dfm), " documents x ", ncol(party_dfm), " features\n"))

In [ ]:
# Step 3: Apply TF-IDF weighting
party_tfidf <- dfm_tfidf(party_dfm)

cat(paste0("TF-IDF matrix dimensions: ", nrow(party_tfidf), " documents x ", ncol(party_tfidf), " features\n"))
party_tfidf

In [ ]:
# Step 4: Identify the top 10 features by average TF-IDF score
mean_tfidf <- colMeans(party_tfidf)
top_features <- sort(mean_tfidf, decreasing = TRUE)[1:10]

cat(paste0("Top 10 features for ", party_name, " speeches by average TF-IDF score:\n"))
for (i in seq_along(top_features)) {
  cat(sprintf("  %-20s %.4f\n", names(top_features)[i], top_features[i]))
}

**Reflection**: The top TF-IDF features for SNP speeches include words like "scotland", "communities", "rural", "fishing", and "scottish" — reflecting the party's focus on Scottish interests, devolution, and the needs of specific communities. These features capture the distinctive thematic priorities of the SNP compared to the broader corpus.

## Bibliography

- Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). *Text as Data: A New Framework for Machine Learning and the Social Sciences*. Princeton University Press.
- Manning, C. D., Raghavan, P., & Schutze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.
- Benoit, K., Watanabe, K., Wang, H., Noel, P., Lori, S., Obeng, A., Muller, S., & Matsuo, A. (2018). quanteda: An R package for the quantitative analysis of textual data. *Journal of Open Source Software*, 3(30), 774.

---

**END OF FILE**